# Emotion Annotation — Final Dataset (N = 800)

This notebook processes the **final** curated subset.
The pilot version is in `thesis-stage3-emotion-annotation__pilot.ipynb`.
Inputs: `datasets/metadata/final_face_manifest.csv`
Outputs: `datasets/emotion_annotated/metadata/final_frame_emotion_predictions.csv`

# Emotion Annotation — Stage 3 (Kaggle-ready)

**Model:** [Empathic-Insight-Face-Large](https://huggingface.co/laion/Empathic-Insight-Face-Large)
(SigLIP2 embeddings → 40 fine-grained emotion MLP heads)

Pipeline:
1. Load `face_manifest.csv`
2. Download Empathic-Insight-Face-Large from HuggingFace
3. Extract SigLIP2 embeddings for each face crop
4. Run 40 emotion MLP heads → per-frame emotion scores
5. Derive valence / arousal from circumplex mapping
6. Save:
   - `frame_emotion_predictions.csv`
   - `video_emotion_features.csv`

## Expected outputs

Frame-level:
- 40 fine-grained emotion scores (mean-subtracted)
- predicted dominant emotion (argmax of scores)
- derived valence (circumplex weighted sum)
- derived arousal (circumplex weighted sum)

Video-level:
- dominant emotion
- mean / std valence
- mean / std arousal
- max arousal
- emotion entropy
- transition rate
- arousal variation
- neutral ratio
- per-emotion mean scores (40 columns)

In [ ]:
%uv install transformers huggingface_hub accelerate -q

In [ ]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
from pathlib import Path
import json
import sys
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from transformers import AutoModel, AutoProcessor
from huggingface_hub import snapshot_download

In [ ]:
# =========================
# Configuration
# =========================
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
input_root = PROJECT_ROOT / Path("datasets")

SUBSET_NAME = "final"   # change to "final" later

FACE_MANIFEST_PATH = input_root / "metadata" / f"{SUBSET_NAME}_face_manifest.csv"
OUT_DIR = input_root / "emotion_annotated" / "metadata"
FRAME_EMOTION_OUT = OUT_DIR / f"{SUBSET_NAME}_frame_emotion_predictions.csv"
VIDEO_FEATURES_OUT = OUT_DIR / f"{SUBSET_NAME}_video_emotion_features.csv"

# Runtime
BATCH_SIZE = 32   # SigLIP2 uses more VRAM than EmoNet; adjust down if OOM
NUM_WORKERS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Model identifiers
HF_EMOTION_REPO = "laion/Empathic-Insight-Face-Large"
SIGLIP_MODEL_ID = "google/siglip2-so400m-patch16-384"  # 1152-dim embeddings
NEUTRAL_STATS_FILE = "neutral_stats_cache-_human-binary-big-mlps.json"

# The 40 emotion categories (EMoNet-FACE taxonomy)
EMOTION_LABELS_40 = [
    "affection", "amusement", "anger", "astonishment_surprise", "awe",
    "bitterness", "concentration", "confusion", "contemplation", "contempt",
    "contentment", "disappointment", "disgust", "distress", "doubt",
    "elation", "embarrassment", "emotional_numbness", "fatigue_exhaustion",
    "fear", "helplessness", "hope_enthusiasm_optimism",
    "impatience_and_irritability", "infatuation", "interest",
    "intoxication_altered_states_of_consciousness", "jealousy_&_envy",
    "longing", "malevolence_malice", "pain", "pleasure_ecstasy", "pride",
    "relief", "sadness", "sexual_lust", "shame", "sourness", "teasing",
    "thankfulness_gratitude", "triumph",
]

# ── Russell's circumplex mapping (valence, arousal) per emotion ──
# Used to derive continuous valence/arousal for downstream compatibility.
EMOTION_VA_MAP: Dict[str, tuple] = {
    "affection":            ( 0.7,  0.3),
    "amusement":            ( 0.8,  0.6),
    "anger":                (-0.7,  0.8),
    "astonishment_surprise":( 0.1,  0.9),
    "awe":                  ( 0.5,  0.5),
    "bitterness":           (-0.6, -0.2),
    "concentration":        ( 0.1,  0.3),
    "confusion":            (-0.2,  0.3),
    "contemplation":        ( 0.1, -0.3),
    "contempt":             (-0.6,  0.2),
    "contentment":          ( 0.7, -0.4),
    "disappointment":       (-0.5, -0.3),
    "disgust":              (-0.7,  0.5),
    "distress":             (-0.7,  0.7),
    "doubt":                (-0.2,  0.1),
    "elation":              ( 0.9,  0.8),
    "embarrassment":        (-0.5,  0.4),
    "emotional_numbness":   (-0.1, -0.7),
    "fatigue_exhaustion":   (-0.3, -0.8),
    "fear":                 (-0.8,  0.8),
    "helplessness":         (-0.6, -0.3),
    "hope_enthusiasm_optimism": (0.7, 0.6),
    "impatience_and_irritability": (-0.5, 0.7),
    "infatuation":          ( 0.6,  0.6),
    "interest":             ( 0.5,  0.4),
    "intoxication_altered_states_of_consciousness": (0.0, -0.2),
    "jealousy_&_envy":      (-0.6,  0.5),
    "longing":              (-0.2, -0.1),
    "malevolence_malice":   (-0.8,  0.6),
    "pain":                 (-0.8,  0.6),
    "pleasure_ecstasy":     ( 0.9,  0.7),
    "pride":                ( 0.6,  0.4),
    "relief":               ( 0.5, -0.4),
    "sadness":              (-0.7, -0.5),
    "sexual_lust":          ( 0.3,  0.7),
    "shame":                (-0.6, -0.2),
    "sourness":             (-0.4,  0.1),
    "teasing":              ( 0.3,  0.4),
    "thankfulness_gratitude": (0.7, 0.2),
    "triumph":              ( 0.8,  0.7),
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FACE_MANIFEST_PATH:", FACE_MANIFEST_PATH)
print("DEVICE:", DEVICE)

In [ ]:
assert FACE_MANIFEST_PATH.exists(), f"Face manifest not found: {FACE_MANIFEST_PATH}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

face_df = pd.read_csv(FACE_MANIFEST_PATH)
print(f"Loaded {len(face_df):,} rows from {FACE_MANIFEST_PATH}")
display(face_df.head())

## Dataset

In [ ]:
class FaceCropDataset:
    """Lightweight wrapper that yields PIL images + metadata in batches."""

    def __init__(self, df: pd.DataFrame, batch_size: int = 32):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size

    def __len__(self):
        return len(self.df)

    def num_batches(self):
        return (len(self.df) + self.batch_size - 1) // self.batch_size

    def iter_batches(self):
        for start in range(0, len(self.df), self.batch_size):
            batch_df = self.df.iloc[start : start + self.batch_size]
            images = []
            meta_rows = []
            for _, row in batch_df.iterrows():
                try:
                    img = Image.open(row["face_path"]).convert("RGB")
                except Exception:
                    continue  # skip unreadable images
                images.append(img)
                meta_rows.append({
                    "video_id": row["video_id"],
                    "frame_id": row["frame_id"],
                    "face_id": row["face_id"],
                    "face_path": row["face_path"],
                    "timestamp_sec": row.get("timestamp_sec", np.nan),
                    "label": row.get("label", None),
                    "split": row.get("split", None),
                    "manipulation_family": row.get("manipulation_family", None),
                    "manipulation_type": row.get("manipulation_type", None),
                })
            if images:
                yield images, meta_rows

dataset = FaceCropDataset(face_df, batch_size=BATCH_SIZE)
print("Dataset size:", len(dataset), f"({dataset.num_batches()} batches)")

In [ ]:
row = face_df.iloc[0]
img = Image.open(row["face_path"]).convert("RGB")

print(row.to_dict())
display(img)

## Load Empathic-Insight-Face-Large

Downloads SigLIP2 backbone and the 40 emotion MLP heads from HuggingFace.
Each MLP takes a 1152-dim SigLIP2 embedding → 1 continuous emotion score.

In [ ]:
# ── MLP architecture (must match the checkpoint) ──
class EmotionMLP(nn.Module):
    def __init__(self, input_size: int = 1152, output_size: int = 1):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 1024), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(1024, 512),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256),         nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, output_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)

# ── Download emotion MLP weights from HuggingFace ──
emotion_repo_dir = Path(snapshot_download(HF_EMOTION_REPO))
print("Emotion repo cached at:", emotion_repo_dir)

# ── Load SigLIP2 ──
siglip_processor = AutoProcessor.from_pretrained(SIGLIP_MODEL_ID)
siglip_model = AutoModel.from_pretrained(SIGLIP_MODEL_ID).to(DEVICE).eval()
print("SigLIP2 loaded")

# ── Load neutral-face statistics for mean subtraction ──
neutral_stats_path = emotion_repo_dir / NEUTRAL_STATS_FILE
neutral_stats: Dict[str, dict] = {}
if neutral_stats_path.exists():
    with open(neutral_stats_path, "r") as f:
        neutral_stats = json.load(f)
    print(f"Loaded neutral stats ({len(neutral_stats)} entries)")
else:
    print(f"WARNING: neutral stats file not found at {neutral_stats_path}")

# ── Load all 40 emotion MLPs ──
emotion_mlps: Dict[str, nn.Module] = {}
for pth_file in sorted(emotion_repo_dir.glob("model_*_best.pth")):
    key = pth_file.stem  # e.g. "model_affection_best"
    mlp = EmotionMLP().to(DEVICE)
    mlp.load_state_dict(torch.load(pth_file, map_location=DEVICE, weights_only=True))
    mlp.eval()
    emotion_mlps[key] = mlp

print(f"Loaded {len(emotion_mlps)} emotion MLP heads")
assert len(emotion_mlps) == 40, f"Expected 40 MLPs, got {len(emotion_mlps)}"

## Inference

In [ ]:
def _normalized(a: np.ndarray, axis: int = -1, order: int = 2) -> np.ndarray:
    """L2-normalise embeddings (same helper used in official example)."""
    norms = np.linalg.norm(a, order, axis, keepdims=True)
    norms[norms == 0] = 1
    return a / norms


def _derive_valence_arousal(
    scores: Dict[str, float],
    va_map: Dict[str, tuple],
) -> tuple:
    """Weighted sum of emotion scores using circumplex coordinates."""
    val, aro, total_weight = 0.0, 0.0, 0.0
    for emo_key, score in scores.items():
        if emo_key in va_map:
            v, a = va_map[emo_key]
            w = max(score, 0.0)  # ignore negative (below-neutral) scores
            val += v * w
            aro += a * w
            total_weight += w
    if total_weight > 0:
        val /= total_weight
        aro /= total_weight
    return float(val), float(aro)


@torch.no_grad()
def run_empathic_inference(
    siglip_model: nn.Module,
    siglip_processor,
    emotion_mlps: Dict[str, nn.Module],
    neutral_stats: Dict[str, dict],
    dataset: FaceCropDataset,
    va_map: Dict[str, tuple],
) -> pd.DataFrame:
    rows: List[dict] = []

    for images, meta_rows in tqdm(
        dataset.iter_batches(),
        total=dataset.num_batches(),
        desc="Running Empathic-Insight-Face-Large",
    ):
        # ── Extract SigLIP2 vision embeddings only (no text input needed) ──
        inputs = siglip_processor(
            images=images, return_tensors="pt", padding="max_length", truncation=True,
        ).to(DEVICE)

        # Use vision_model directly — avoids requiring input_ids from full SiglipModel
        vision_output = siglip_model.vision_model(pixel_values=inputs["pixel_values"])
        embeddings = vision_output.pooler_output  # (B, 1152)
        embeddings_np = _normalized(embeddings.cpu().numpy())
        embeddings_t = torch.from_numpy(embeddings_np).to(DEVICE).float()

        # ── Run each emotion MLP ──
        all_scores: Dict[str, np.ndarray] = {}
        for model_key, mlp in emotion_mlps.items():
            raw = mlp(embeddings_t).cpu().numpy().reshape(-1)
            neutral_mean = neutral_stats.get(model_key, {}).get("mean", 0.0)
            all_scores[model_key] = raw - neutral_mean

        # ── Build per-sample rows ──
        for i, meta in enumerate(meta_rows):
            sample_scores = {}
            for model_key, score_arr in all_scores.items():
                emo_name = model_key.replace("model_", "").replace("_best", "")
                sample_scores[emo_name] = float(score_arr[i])

            pred_emotion = max(sample_scores, key=sample_scores.get)
            pred_score = sample_scores[pred_emotion]
            valence, arousal = _derive_valence_arousal(sample_scores, va_map)

            row = {
                **meta,
                "pred_emotion": pred_emotion,
                "pred_emotion_score": pred_score,
                "valence": valence,
                "arousal": arousal,
            }
            for emo_name, sc in sample_scores.items():
                row[f"score_{emo_name}"] = sc

            rows.append(row)

    return pd.DataFrame(rows)


# ── Run inference ──────────────────────────────────────────────────────────────
frame_emotion_df = run_empathic_inference(
    siglip_model, siglip_processor,
    emotion_mlps, neutral_stats,
    dataset, EMOTION_VA_MAP,
)
print(f"Generated {len(frame_emotion_df):,} frame-level predictions")
display(frame_emotion_df.head())

frame_emotion_df.to_csv(FRAME_EMOTION_OUT, index=False)
print("Saved:", FRAME_EMOTION_OUT)


In [ ]:
def _normalized(a: np.ndarray, axis: int = -1, order: int = 2) -> np.ndarray:
    """L2-normalise embeddings (same helper used in official example)."""
    norms = np.linalg.norm(a, order, axis, keepdims=True)
    norms[norms == 0] = 1
    return a / norms


def _derive_valence_arousal(
    scores: Dict[str, float],
    va_map: Dict[str, tuple],
) -> tuple:
    """Weighted sum of emotion scores using circumplex coordinates."""
    val, aro, total_weight = 0.0, 0.0, 0.0
    for emo_key, score in scores.items():
        if emo_key in va_map:
            v, a = va_map[emo_key]
            w = max(score, 0.0)  # ignore negative (below-neutral) scores
            val += v * w
            aro += a * w
            total_weight += w
    if total_weight > 0:
        val /= total_weight
        aro /= total_weight
    return float(val), float(aro)


@torch.no_grad()
def run_empathic_inference(
    siglip_model: nn.Module,
    siglip_processor,
    emotion_mlps: Dict[str, nn.Module],
    neutral_stats: Dict[str, dict],
    dataset: FaceCropDataset,
    va_map: Dict[str, tuple],
) -> pd.DataFrame:
    rows: List[dict] = []

    for images, meta_rows in tqdm(
        dataset.iter_batches(),
        total=dataset.num_batches(),
        desc="Running Empathic-Insight-Face-Large",
    ):
        # ── Extract SigLIP2 embeddings ──
        inputs = siglip_processor(
            images=images, return_tensors="pt", padding="max_length", truncation=True,
        ).to(DEVICE)
        
        # Get embeddings from model output
        model_output = siglip_model(**inputs)
        # SigLIP returns BaseModelOutputWithPooling, extract pooler_output (image embeddings)
        embeddings = model_output.pooler_output  # (B, 1152)
        embeddings_np = _normalized(embeddings.cpu().numpy())
        embeddings_t = torch.from_numpy(embeddings_np).to(DEVICE).float()

        # ── Run each emotion MLP ──
        # shape: {model_key: (B,)} raw scores
        all_scores: Dict[str, np.ndarray] = {}
        for model_key, mlp in emotion_mlps.items():
            raw = mlp(embeddings_t).cpu().numpy().reshape(-1)
            neutral_mean = neutral_stats.get(model_key, {}).get("mean", 0.0)
            all_scores[model_key] = raw - neutral_mean

        # ── Build per-sample rows ──
        for i, meta in enumerate(meta_rows):
            # Collect per-emotion scores for this sample
            sample_scores = {}
            for model_key, score_arr in all_scores.items():
                emo_name = model_key.replace("model_", "").replace("_best", "")
                sample_scores[emo_name] = float(score_arr[i])

            # Dominant emotion = highest mean-subtracted score
            pred_emotion = max(sample_scores, key=sample_scores.get)
            pred_score = sample_scores[pred_emotion]

            # Derive valence / arousal from circumplex mapping
            valence, arousal = _derive_valence_arousal(sample_scores, va_map)

            row = {
                **meta,
                "pred_emotion": pred_emotion,
                "pred_emotion_score": pred_score,
                "valence": valence,
                "arousal": arousal,
            }
            # Add all 40 per-emotion scores as score_<name> columns
            for emo_name, sc in sample_scores.items():
                row[f"score_{emo_name}"] = sc

            rows.append(row)

    return pd.DataFrame(rows)

## Aggregate to video-level features

In [ ]:
# Load frame_emotion_df — from memory if available, else from saved CSV
if 'frame_emotion_df' not in dir() or not isinstance(frame_emotion_df, pd.DataFrame):
    if FRAME_EMOTION_OUT.exists():
        frame_emotion_df = pd.read_csv(FRAME_EMOTION_OUT)
        print(f"Loaded frame_emotion_df from CSV: {FRAME_EMOTION_OUT}")
    else:
        raise FileNotFoundError(
            f"frame_emotion_df not in memory and no saved CSV found at:\n  {FRAME_EMOTION_OUT}\n"
            "Please run the inference cell first."
        )

print(f"✓ frame_emotion_df ready: {len(frame_emotion_df):,} rows, {len(frame_emotion_df.columns)} columns")


In [ ]:
def compute_entropy(labels: List[str]) -> float:
    if len(labels) == 0:
        return np.nan
    _, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()
    return float(-(probs * np.log(probs + 1e-12)).sum())

def compute_transition_rate(labels: List[str]) -> float:
    if len(labels) <= 1:
        return 0.0
    changes = sum(labels[i] != labels[i - 1] for i in range(1, len(labels)))
    return float(changes / (len(labels) - 1))

def compute_arousal_variation(arousal_values: np.ndarray) -> float:
    if len(arousal_values) <= 1:
        return 0.0
    return float(np.mean(np.abs(np.diff(arousal_values))))


def aggregate_video_features(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    
    # Detect score_* columns from the dataframe
    score_cols = [c for c in df.columns if c.startswith("score_")]

    for video_id, grp in tqdm(df.groupby("video_id"), desc="Aggregating by video"):
        grp = grp.sort_values("timestamp_sec").reset_index(drop=True)

        emotions = grp["pred_emotion"].tolist()
        valence = grp["valence"].to_numpy()
        arousal = grp["arousal"].to_numpy()

        dominant_emotion = grp["pred_emotion"].mode().iloc[0] if len(grp) else None
        neutral_ratio = float(
            (grp["pred_emotion"].isin(["emotional_numbness", "contentment", "contemplation"])).mean()
        ) if len(grp) else np.nan

        row = {
            "video_id": video_id,
            "label": grp["label"].iloc[0],
            "split": grp["split"].iloc[0],
            "manipulation_family": grp["manipulation_family"].iloc[0],
            "manipulation_type": grp["manipulation_type"].iloc[0],
            "n_face_frames": len(grp),
            "dominant_emotion": dominant_emotion,
            "mean_valence": float(np.mean(valence)),
            "std_valence": float(np.std(valence)),
            "mean_arousal": float(np.mean(arousal)),
            "std_arousal": float(np.std(arousal)),
            "max_arousal": float(np.max(arousal)),
            "emotion_entropy": compute_entropy(emotions),
            "transition_rate": compute_transition_rate(emotions),
            "arousal_variation": compute_arousal_variation(arousal),
            "neutral_ratio": neutral_ratio,
        }

        # Per-emotion mean scores (40 extra columns)
        for sc in score_cols:
            row[f"mean_{sc}"] = float(grp[sc].mean())

        rows.append(row)

    return pd.DataFrame(rows)

video_features_df = aggregate_video_features(frame_emotion_df)
print(f"Generated {len(video_features_df):,} video-level rows")
display(video_features_df.head())

video_features_df.to_csv(VIDEO_FEATURES_OUT, index=False)
print("Saved:", VIDEO_FEATURES_OUT)

In [ ]:
print("Frame-level rows:", len(frame_emotion_df))
print("Video-level rows:", len(video_features_df))
display(video_features_df[["label", "split"]].value_counts().rename("count").reset_index())

if "dominant_emotion" in video_features_df.columns:
    display(video_features_df["dominant_emotion"].value_counts().rename_axis("dominant_emotion").reset_index(name="count"))

## Next stage

After this notebook:
1. Run baseline deepfake detector to obtain `detector_score`
2. Merge:
   - manifest
   - video emotion features
   - detector scores
3. Run experiments:
   - baseline only
   - emotion only
   - fusion model
   - ablation